# 1.1 EDA — Sine Sweep (5–500 Hz, 0.2 g)
**Data:** Lenovo SR650 V2 shaker table sinesweep

In [42]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.signal import find_peaks

CSV = "1.1 - Data and Plots/Lenovo SR650 V2 Table Sinesweep 5hz 500hz 0.2g.csv"

df = pd.read_csv(CSV, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

# Drop trailing empty rows and rows with no frequency
df = df[pd.to_numeric(df["Frequency (Hz)"], errors="coerce").notna()].copy()
df = df.apply(pd.to_numeric, errors="coerce")
df = df.reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Frequency range: {df['Frequency (Hz)'].min():.1f} – {df['Frequency (Hz)'].max():.1f} Hz")
df.head(3)

Shape: (2000, 203)
Frequency range: 5.0 – 500.0 Hz


,Frequency (Hz),Demand1 (G),Control1 (G),Abort1(+),Tol1(+),Channel Tol1(+),Channel Tol1(-),Tol1(-),Abort1(-),Control1 (Prev) (G),...,CPU1 (G).7,CPU2 (G).2,CPU1 (G).8,CPU0 (G).5,BL Switch 32Z (G).2,BR Switch 2X (G).2,BR Switch 2Y (G).2,BR Switch 2Z (G).2,BR Switch 4 (G).2,BR Switch 16 (G).2
0,5.00000,0.2,0.195983,0.399052,0.282508,0.282508,0.141589,0.141589,0.100237,0.0,...,0.002466,-0.000121,0.000054,0.001213,0.004421,-0.001198,-0.001436,-0.001323,0.001630,0.000155
1,5.01153,0.2,0.196025,0.399052,0.282508,0.282508,0.141589,0.141589,0.100237,0.0,...,0.000476,-0.001052,-0.000869,-0.000252,-0.001195,-0.001198,0.001635,0.000226,-0.001090,-0.000340
2,5.02309,0.2,0.196064,0.399052,0.282508,0.282508,0.141589,0.141589,0.100237,0.0,...,0.001405,-0.003579,0.001768,-0.001849,-0.000373,0.002100,-0.001436,0.000872,0.001888,-0.000418


## Data Overview

In [43]:
# --- Data overview ---
print("=== Null counts (columns with any nulls) ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "None")

print("\n=== Acceleration (G) columns ===")
accel_cols = [c for c in df.columns if "(G)" in c
              and not any(x in c for x in ["Abort", "Tol", "Channel", "Prev", "Demand"])]
print(accel_cols)

print("\n=== Summary stats for key acceleration channels ===")
key_channels = ["Table (G)", "PCB_0 (G)", "PCB_Mid (G)", "PCB_1 (G)"]
df[key_channels].describe().round(4)

=== Null counts (columns with any nulls) ===
None

=== Acceleration (G) columns ===
['Control1 (G)', 'Table (G)', 'PCB_0 (G)', 'PCB_Mid (G)', 'PCB_1 (G)', 'CPU0 (G)', 'CPU1 (G)', 'CPU1 (G).1', 'CPU2 (G)', 'CPU1 (G).2', 'CPU0 (G).1', 'BL Switch 32Z (G)', 'BR Switch 2X (G)', 'BR Switch 2Y (G)', 'BR Switch 2Z (G)', 'BR Switch 4 (G)', 'BR Switch 16 (G)', 'Control1 (G).1', 'Table (G).1', 'PCB_0 (G).1', 'PCB_Mid (G).1', 'PCB_1 (G).1', 'CPU0 (G).2', 'CPU1 (G).3', 'CPU1 (G).4', 'CPU2 (G).1', 'CPU1 (G).5', 'CPU0 (G).3', 'BL Switch 32Z (G).1', 'BR Switch 2X (G).1', 'BR Switch 2Y (G).1', 'BR Switch 2Z (G).1', 'BR Switch 4 (G).1', 'BR Switch 16 (G).1', 'Table (G).2', 'PCB_0 (G).2', 'PCB_Mid (G).2', 'PCB_1 (G).2', 'CPU0 (G).4', 'CPU1 (G).6', 'CPU1 (G).7', 'CPU2 (G).2', 'CPU1 (G).8', 'CPU0 (G).5', 'BL Switch 32Z (G).2', 'BR Switch 2X (G).2', 'BR Switch 2Y (G).2', 'BR Switch 2Z (G).2', 'BR Switch 4 (G).2', 'BR Switch 16 (G).2']

=== Summary stats for key acceleration channels ===


,Table (G),PCB_0 (G),PCB_Mid (G),PCB_1 (G)
count,2000.0000,2000.0000,2000.0000,2000.0000
mean,0.1995,0.3312,0.2931,0.2426
std,0.0023,0.4491,0.1849,0.1513
min,0.1877,0.0121,0.0123,0.0079
25%,0.1986,0.1205,0.1841,0.1295
50%,0.1995,0.2238,0.2449,0.2187
75%,0.2003,0.3774,0.3599,0.2871
max,0.2237,4.0450,0.8972,0.7554


## Acceleration Response

In [44]:
# --- Acceleration response vs frequency ---
freq = df["Frequency (Hz)"]
channels = {
    "Table": "Table (G)",
    "PCB_0": "PCB_0 (G)",
    "PCB_Mid": "PCB_Mid (G)",
    "PCB_1": "PCB_1 (G)",
    "Control": "Control1 (G)",
}

fig = go.Figure()
for label, col in channels.items():
    if col in df.columns:
        fig.add_trace(go.Scatter(x=freq, y=df[col].abs(), name=label, mode="lines", line=dict(width=1.2)))

fig.add_hline(y=0.2, line_dash="dash", line_color="black", annotation_text="Demand (0.2 g)")

fig.update_layout(
    title="Sine Sweep Acceleration Response",
    xaxis=dict(title="Frequency (Hz)", type="log"),
    yaxis=dict(title="Acceleration (G)", type="log"),
    legend=dict(orientation="v"),
    height=500,
)
fig.show()

In [53]:
# --- Transmissibility (sensor / table) ---
table = df["Table (G)"].replace(0, np.nan)

t_channels = {
    "PCB_0":   "PCB_0 (G)",
    "PCB_Mid": "PCB_Mid (G)",
    "PCB_1":   "PCB_1 (G)",
    "CPU0":    "CPU0 (G)",
    "CPU1":    "CPU1 (G)",
}

transmissibility = {}
for label, col in t_channels.items():
    if col in df.columns:
        T = (df[col] / table).abs()
        # Clip to physically meaningful range (table near-zero at sweep edges causes blow-up)
        transmissibility[label] = T.clip(upper=100)

# Identify major peaks from PCB_0 (highest amplitude, best reference)
T_ref = transmissibility["PCB_0"].fillna(0).values
top_peaks, _ = find_peaks(T_ref, height=2.0, prominence=0.8, distance=50)
top_peaks = sorted(top_peaks, key=lambda p: T_ref[p], reverse=True)[:5]

fig = go.Figure()
for label, T in transmissibility.items():
    fig.add_trace(go.Scatter(x=freq, y=T, name=label, mode="lines", line=dict(width=1.2)))

fig.add_hline(y=1.0, line_dash="dash", line_color="black")

fig.update_layout(
    title="Transmissibility (reference = Table)",
    xaxis=dict(title="Frequency (Hz)", type="log", range=[np.log10(5), np.log10(500)]),
    yaxis=dict(title="Transmissibility (G/G)", type="log", range=[-1, 2]),  # 0.1 to 100
    height=550,
)
fig.show()

## Resonance Peaks
Finding peaks in frequency bands

In [47]:
# --- All frequencies with transmissibility > 2 ---
rows = []
for label, T in transmissibility.items():
    T_clean = T.fillna(0).values
    peaks, _ = find_peaks(T_clean, height=2.0, distance=20)
    for p in peaks:
        rows.append({"Channel": label, "Frequency (Hz)": round(freq.iloc[p], 1), "Transmissibility": round(T_clean[p], 2)})

display(pd.DataFrame(rows))

# --- Natural frequency bands ---
nf_bands = {
    "1st NF  (~28 Hz)":   (20, 40),
    "2nd NF  (~60 Hz)":   (50, 80),
    "Additional (>80 Hz)": (80, 600),
}

# Use PCB_0 as reference — highest amplitude across all bands
T_ref_clean = transmissibility["PCB_0"].fillna(0).values
all_peaks, _ = find_peaks(T_ref_clean, height=1.5, prominence=0.5, distance=30)

summary = []
for band_name, (f_lo, f_hi) in nf_bands.items():
    band_peaks = [p for p in all_peaks if f_lo <= freq.iloc[p] <= f_hi]
    if not band_peaks:
        continue
    best = max(band_peaks, key=lambda p: T_ref_clean[p])
    # Collect T values at that frequency across all channels
    row = {"Natural Frequency": band_name, "Frequency (Hz)": round(float(freq.iloc[best]), 2)}
    for label, T in transmissibility.items():
        row[f"T – {label}"] = round(float(T.fillna(0).iloc[best]), 2)
    summary.append(row)

print("\nNatural Frequency Summary:")
display(pd.DataFrame(summary))

,Channel,Frequency (Hz),Transmissibility
0,PCB_0,23.1,2.73
1,PCB_0,25.3,3.47
2,PCB_0,27.2,4.11
3,PCB_0,28.5,4.35
4,PCB_0,29.9,3.60
...,...,...,...
59,CPU1,28.0,3.50
60,CPU1,29.5,3.21
61,CPU1,31.2,2.46
62,CPU1,35.4,2.04



Natural Frequency Summary:


,Natural Frequency,Frequency (Hz),T – PCB_0,T – PCB_Mid,T – PCB_1,T – CPU0,T – CPU1
0,1st NF (~28 Hz),28.53,4.35,4.53,3.81,4.04,3.49
1,Additional (>80 Hz),487.49,20.75,1.44,1.28,1.71,0.91
